# Lesson 32: ML Pipeline
**Module 6: ML Unsupervised**

Interactive notebook for building ML pipelines.

## Learning Objectives
- Build sklearn pipelines
- Use ColumnTransformer
- Combine with GridSearchCV
- Save and load pipelines

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
np.random.seed(42)

# Sample mixed data
n = 200
df = pd.DataFrame({
    'income': np.random.exponential(50000, n),
    'age': np.random.randint(25, 65, n),
    'sector': np.random.choice(['Tech', 'Finance', 'Health'], n),
    'approved': np.random.randint(0, 2, n)
})
# Add some missing values
df.loc[np.random.choice(n, 10), 'income'] = np.nan

X = df.drop('approved', axis=1)
y = df['approved']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print('Data ready!')

## 1. Simple Pipeline

In [ ]:
# Numeric-only pipeline
numeric_cols = ['income', 'age']

simple_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

simple_pipe.fit(X_train[numeric_cols], y_train)
print(f'Accuracy: {simple_pipe.score(X_test[numeric_cols], y_test):.3f}')

## 2. ColumnTransformer for Mixed Data

In [ ]:
# Define transformers for each column type
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, ['income', 'age']),
    ('cat', categorical_transformer, ['sector'])
])

# Full pipeline
full_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

full_pipe.fit(X_train, y_train)
print(f'Full pipeline accuracy: {full_pipe.score(X_test, y_test):.3f}')

## 3. GridSearchCV with Pipeline

In [ ]:
# Tune model parameter through pipeline
param_grid = {'model__C': [0.1, 1.0, 10.0]}

grid = GridSearchCV(full_pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(f'Best C: {grid.best_params_}')
print(f'Best CV score: {grid.best_score_:.3f}')
print(f'Test score: {grid.score(X_test, y_test):.3f}')

## Exercise: Save and Load Pipeline
Save the trained pipeline to disk.

In [ ]:
# Exercise: Save and reload the pipeline
# 1. Use joblib.dump() to save
# 2. Use joblib.load() to reload
# 3. Verify loaded model gives same predictions

# YOUR CODE HERE

---
## Solution
<details>
<summary>Click to reveal</summary>

```python
import joblib

# Save
joblib.dump(full_pipe, 'credit_model.pkl')
print('Model saved!')

# Load
loaded_pipe = joblib.load('credit_model.pkl')
print('Model loaded!')

# Verify
original_pred = full_pipe.predict(X_test)
loaded_pred = loaded_pipe.predict(X_test)
print(f'Predictions match: {(original_pred == loaded_pred).all()}')
```
</details>